# Notebook 07: SVMs and Kernel Methods

## Why This Matters

Support Vector Machines were the dominant ML algorithm before deep learning and gradient boosting took over, and they remain important for several reasons. First, SVMs with RBF kernels are still competitive on small-to-medium datasets where tree ensembles may overfit. Second, the mathematical framework — maximum-margin hyperplanes, the dual problem, the kernel trick — is intellectually foundational and appears consistently in advanced ML interviews. Third, the kernel trick introduces the profound idea that you can operate in an infinite-dimensional feature space implicitly, which directly connects to neural network theory (neural tangent kernels) and Gaussian processes. Understanding SVMs deeply signals strong theoretical ML foundations.

## 1. The Maximum Margin Classifier

For a linearly separable binary classification problem, many hyperplanes can separate the classes. SVMs choose the one that **maximizes the margin** — the distance between the hyperplane and the nearest training points (support vectors).

**Primal problem:**
$$\min_{\mathbf{w}, b} \frac{1}{2}\|\mathbf{w}\|^2 \quad \text{subject to} \quad y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 \; \forall i$$

The margin is $\frac{2}{\|\mathbf{w}\|}$, so minimizing $\|\mathbf{w}\|^2$ maximizes the margin.

**Support vectors**: training points that lie exactly on the margin boundaries ($y_i(\mathbf{w}^T \mathbf{x}_i + b) = 1$). Only these points matter for the solution — all other training points can be removed without changing the decision boundary. This is a key property that makes SVMs memory-efficient at inference time.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC, SVR
from sklearn.datasets import make_classification, make_circles, make_moons, load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline

# Visualize maximum margin classifier in 2D
np.random.seed(42)
X_lin = np.random.randn(80, 2)
y_lin = (X_lin[:, 0] * 0.8 + X_lin[:, 1] * 0.6 > 0).astype(int) * 2 - 1

svm_lin = SVC(kernel='linear', C=1000)  # Large C ≈ hard margin
svm_lin.fit(X_lin, y_lin)

fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(X_lin[y_lin==1, 0], X_lin[y_lin==1, 1],
           c='steelblue', s=40, zorder=5, label='Class +1')
ax.scatter(X_lin[y_lin==-1, 0], X_lin[y_lin==-1, 1],
           c='coral', s=40, zorder=5, label='Class -1')

# Decision boundary and margins
xx = np.linspace(X_lin[:, 0].min() - 0.5, X_lin[:, 0].max() + 0.5, 100)
w = svm_lin.coef_[0]
b = svm_lin.intercept_[0]
for bias, style, lbl in [(0, '-', 'Decision boundary'), (1, '--', 'Margin'), (-1, '--', None)]:
    yy = -(w[0] * xx + b + bias) / w[1]
    ax.plot(xx, yy, 'k' + style, linewidth=1.5, label=lbl)

# Highlight support vectors
sv = svm_lin.support_vectors_
ax.scatter(sv[:, 0], sv[:, 1], s=150, facecolors='none', edgecolors='black',
           linewidths=2, zorder=6, label=f'Support vectors ({len(sv)})')

margin_width = 2 / np.linalg.norm(w)
ax.set_title(f'Linear SVM — Maximum Margin = {margin_width:.3f}', fontsize=11)
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

## 2. Soft Margin SVM: The C Parameter

Real data is rarely linearly separable. The **soft-margin SVM** introduces slack variables $\xi_i \geq 0$ that allow some points to be inside the margin or on the wrong side:

$$\min_{\mathbf{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\mathbf{w}\|^2 + C \sum_i \xi_i \quad \text{s.t.} \quad y_i(\mathbf{w}^T \mathbf{x}_i + b) \geq 1 - \xi_i, \; \xi_i \geq 0$$

**C** is the regularization parameter (inverse of λ):
- **Large C**: penalize violations heavily → small margin, fewer misclassifications in training → risk of overfitting
- **Small C**: allow many violations → large margin, more robust → may underfit

The loss function this minimizes is equivalent to **hinge loss + L2 penalty** on w:
$$\mathcal{L} = C \sum_i \max(0, 1 - y_i (\mathbf{w}^T \mathbf{x}_i + b)) + \frac{1}{2}\|\mathbf{w}\|^2$$

In [ ]:
# Effect of C on margin width and decision boundary
X_soft, y_soft = make_classification(n_samples=100, n_features=2, n_redundant=0,
                                      n_informative=2, n_clusters_per_class=1,
                                      class_sep=0.5, random_state=42)
y_soft = y_soft * 2 - 1  # convert to +1/-1

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, C in zip(axes, [0.01, 0.1, 1, 100]):
    svm = SVC(kernel='linear', C=C).fit(X_soft, y_soft)
    xx, yy = np.meshgrid(
        np.linspace(X_soft[:, 0].min()-0.5, X_soft[:, 0].max()+0.5, 200),
        np.linspace(X_soft[:, 1].min()-0.5, X_soft[:, 1].max()+0.5, 200)
    )
    Z = svm.decision_function(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, levels=[-2, -1, 0, 1, 2], alpha=0.2,
                colors=['coral', 'lightyellow', 'lightyellow', 'steelblue'])
    ax.contour(xx, yy, Z, levels=[-1, 0, 1], colors=['red', 'black', 'blue'],
               linestyles=['--', '-', '--'], linewidths=[1, 2, 1])
    ax.scatter(X_soft[:, 0], X_soft[:, 1], c=y_soft, cmap='bwr', s=20, edgecolors='k', lw=0.3)
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none', edgecolors='k', lw=2)
    w = svm.coef_[0]
    margin = 2 / np.linalg.norm(w)
    ax.set_title(f'C={C}\nMargin={margin:.2f}  SVs={len(sv)}', fontsize=9)

plt.suptitle('Soft-Margin SVM: Effect of C', fontsize=12)
plt.tight_layout()
plt.show()

## 3. The Kernel Trick

For non-linearly separable data, we can map features to a higher-dimensional space φ(x) where the data becomes linearly separable. The key insight: in the dual SVM formulation, inputs appear only in dot products $\langle \phi(x_i), \phi(x_j) \rangle$. If we can compute this dot product without explicitly computing φ(x) — using a **kernel function** $K(x_i, x_j) = \langle \phi(x_i), \phi(x_j) \rangle$ — we never need to construct the high-dimensional representation.

**Common kernels:**

| Kernel | Formula | φ dimension | Use case |
|--------|---------|-------------|----------|
| Linear | $\mathbf{x}_i^T \mathbf{x}_j$ | d (original) | Linearly separable; text classification |
| Polynomial | $(\mathbf{x}_i^T \mathbf{x}_j + c)^d$ | O(n^d) | Degree-d interaction features |
| RBF (Gaussian) | $\exp(-\gamma \|\mathbf{x}_i - \mathbf{x}_j\|^2)$ | ∞ | Most problems; default choice |
| Sigmoid | $\tanh(\gamma \mathbf{x}_i^T \mathbf{x}_j + c)$ | — | Rarely used; not always PSD |

In [ ]:
# Visualize kernel SVM on non-linearly separable datasets
datasets = [
    ('Circles', *make_circles(n_samples=300, noise=0.1, factor=0.4, random_state=42)),
    ('Moons',   *make_moons(n_samples=300, noise=0.1, random_state=42)),
    ('XOR',     None, None),  # synthetic XOR
]

# Create XOR dataset
np.random.seed(42)
X_xor = np.random.randn(200, 2)
y_xor = np.logical_xor(X_xor[:, 0] > 0, X_xor[:, 1] > 0).astype(int)
datasets[2] = ('XOR', X_xor, y_xor)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
for col, (name, X_d, y_d) in enumerate(datasets):
    sc = StandardScaler()
    X_d_s = sc.fit_transform(X_d)
    
    for row, (kernel, params) in enumerate([
        ('linear', {'C': 1}),
        ('rbf',    {'C': 1, 'gamma': 'scale'}),
    ]):
        ax = axes[row][col]
        svm = SVC(kernel=kernel, **params).fit(X_d_s, y_d)
        acc = svm.score(X_d_s, y_d)
        
        xx, yy = np.meshgrid(
            np.linspace(X_d_s[:, 0].min()-0.5, X_d_s[:, 0].max()+0.5, 150),
            np.linspace(X_d_s[:, 1].min()-0.5, X_d_s[:, 1].max()+0.5, 150)
        )
        Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
        ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
        ax.scatter(X_d_s[:, 0], X_d_s[:, 1], c=y_d, cmap='bwr', s=15, edgecolors='k', lw=0.3)
        ax.set_title(f'{name} | {kernel} kernel\nacc={acc:.2f}', fontsize=9)

plt.suptitle('Linear vs. RBF Kernel: Non-Linear Decision Boundaries', fontsize=12)
plt.tight_layout()
plt.show()

## 4. RBF Kernel: Understanding Gamma

The RBF kernel $K(x_i, x_j) = \exp(-\gamma \|x_i - x_j\|^2)$ has one parameter: **γ** (gamma).

- **Large γ**: the kernel drops off quickly → each support vector has only local influence → model is very wiggly, fits training data tightly → overfitting
- **Small γ**: the kernel is broad → each support vector has global influence → smoother decision boundary → may underfit

Intuition: γ = 1/(2σ²) where σ is the "radius" of influence. Large γ = small radius = local model. Small γ = large radius = global model.

In practice, tune C and γ jointly with grid search or random search (they interact).

In [ ]:
# Effect of gamma on RBF SVM decision boundary
X_c, y_c = make_circles(n_samples=200, noise=0.15, factor=0.4, random_state=42)
X_c_s = StandardScaler().fit_transform(X_c)

fig, axes = plt.subplots(1, 5, figsize=(18, 4))
for ax, gamma in zip(axes, [0.01, 0.1, 1, 10, 100]):
    svm = SVC(kernel='rbf', C=1, gamma=gamma).fit(X_c_s, y_c)
    xx, yy = np.meshgrid(
        np.linspace(-2.5, 2.5, 150), np.linspace(-2.5, 2.5, 150)
    )
    Z = svm.predict(np.c_[xx.ravel(), yy.ravel()]).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
    ax.scatter(X_c_s[:, 0], X_c_s[:, 1], c=y_c, cmap='bwr', s=20, edgecolors='k', lw=0.3)
    tr_acc = svm.score(X_c_s, y_c)
    ax.set_title(f'γ={gamma}\ntrain_acc={tr_acc:.2f}', fontsize=9)

plt.suptitle('RBF SVM: Effect of Gamma (C=1 fixed)', fontsize=12)
plt.tight_layout()
plt.show()

## 5. Scaling is Essential for SVMs

Unlike tree-based models, SVMs are extremely sensitive to feature scale. The RBF kernel computes Euclidean distances between points — a feature measured in thousands (e.g., income) will dominate the distance calculation and render all other features irrelevant. Always scale before fitting an SVM.

In [ ]:
from sklearn.datasets import load_breast_cancer

data = load_breast_cancer()
Xbc, ybc = data.data, data.target
Xbc_tr, Xbc_te, ybc_tr, ybc_te = train_test_split(Xbc, ybc, test_size=0.2, random_state=42)

# SVM without scaling
svm_noscale = SVC(kernel='rbf', probability=True, random_state=42)
svm_noscale.fit(Xbc_tr, ybc_tr)
auc_noscale = roc_auc_score(ybc_te, svm_noscale.predict_proba(Xbc_te)[:, 1])

# SVM with scaling (pipeline prevents data leakage)
pipe_svm = Pipeline([
    ('scale', StandardScaler()),
    ('svm', SVC(kernel='rbf', probability=True, random_state=42))
])
pipe_svm.fit(Xbc_tr, ybc_tr)
auc_scaled = roc_auc_score(ybc_te, pipe_svm.predict_proba(Xbc_te)[:, 1])

print(f"SVM without scaling: AUC = {auc_noscale:.4f}")
print(f"SVM with scaling:    AUC = {auc_scaled:.4f}")
print("\nDifference can be enormous in practice — always use a pipeline!")

In [ ]:
# Grid search over C and gamma
param_grid = {
    'svm__C': [0.01, 0.1, 1, 10, 100],
    'svm__gamma': [0.001, 0.01, 0.1, 1, 'scale']
}
grid = GridSearchCV(pipe_svm, param_grid, cv=5, scoring='roc_auc', n_jobs=-1)
grid.fit(Xbc_tr, ybc_tr)

print(f"Best params: {grid.best_params_}")
print(f"Best CV AUC: {grid.best_score_:.4f}")
print(f"Test AUC:    {roc_auc_score(ybc_te, grid.predict_proba(Xbc_te)[:, 1]):.4f}")

## 6. SVMs vs. Other Algorithms: When to Use

SVMs remain relevant in specific scenarios:
- Small-to-medium datasets (n < ~10K) where training cost O(n²–n³) is acceptable
- High-dimensional, low-sample problems (text, gene expression) — linear SVM excels here
- Problems requiring a probabilistic output (SVC with `probability=True`, uses Platt scaling)
- When you need a theoretically grounded method with margin guarantees

## Summary

| Concept | Key Insight | Interview Point |
|---------|-------------|------------------|
| Maximum margin | SVM chooses the hyperplane maximizing distance to nearest training points | Only support vectors define the decision boundary |
| Soft margin (C) | C trades off margin width vs. training error | Large C = small margin = overfit; small C = large margin = underfit |
| Kernel trick | Implicitly map to infinite-dimensional space via K(xi, xj) | Never construct φ(x) explicitly; compute dot products in transformed space |
| RBF kernel gamma | Controls locality of influence | Large γ → overfit; small γ → underfit; tune jointly with C |
| Scale sensitivity | Distance-based → must scale features | Always use StandardScaler in pipeline before SVM |
| SVM vs. GBM | SVM: margin-based; great for high-d/small-n; no native feature importance | GBM: faster training at scale; native feature importance; handles mixed types |